<a href="https://colab.research.google.com/github/akbar527/Akbar/blob/Akbar527/1_RecA_ChEMBL_EC50_combined_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RecA–TB ChEMBL EC50 Workflow

Notebook ini menggabungkan tiga script Python menjadi satu workflow yang dapat dijalankan ulang secara berurutan:

1. Download data target RecA dari ChEMBL.
2. Processing data EC50 dan binary label.
3. Data curation final, labeling, dan balancing dataset.

## 0. Instalasi package yang dibutuhkan

Jalankan cell ini jika `chembl_webresource_client` belum tersedia di environment notebook Anda.

In [1]:
# Uncomment jika package belum terinstal
!pip install chembl_webresource_client pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.6 MB/s eta 0:00:00


In [2]:
!pip install pandas numpy scikit-learn matplotlib xgboost shap lime catboost chembl_webresource_client padelpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 72.0 MB/s eta 0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=908ec497278d724596cfdd806a1bda9b8f46706b230857767f8f9de098444cf7
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


## 1. ChEMBL Download Workflow

Bagian ini mencari target **Mycobacterium tuberculosis RecA (CHEMBL1741171)**, mengambil data bioaktivitas, memfilter EC50 exact dalam nM, menghitung pEC50, mengambil metadata molekul, lalu menyimpan output ke folder `outputs/recA_chembl`.

In [3]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Iterable

import pandas as pd
from chembl_webresource_client.new_client import new_client


# ============================================================
# TARGET CONFIGURATION
# ============================================================

TARGET_SEARCH_QUERY = "RecA Mycobacterium tuberculosis"
TARGET_CHEMBL_ID = "CHEMBL1741171"
TARGET_PREF_NAME = "Protein RecA"
TARGET_ORGANISM = "Mycobacterium tuberculosis"

MOLECULE_BATCH_SIZE = 100


# ============================================================
# SEARCH TARGET
# ============================================================

def search_targets(query: str) -> pd.DataFrame:
    """Search ChEMBL targets using query."""
    hits = list(new_client.target.search(query))

    if not hits:
        raise RuntimeError(
            f"No targets found for query: {query}"
        )

    return pd.DataFrame(hits)


def select_target(target_hits: pd.DataFrame) -> pd.Series:
    """Select the expected MtRecA target."""

    match = target_hits[
        (target_hits["target_chembl_id"] == TARGET_CHEMBL_ID)
        |
        (
            (target_hits["pref_name"] == TARGET_PREF_NAME)
            &
            (target_hits["organism"] == TARGET_ORGANISM)
        )
    ]

    if match.empty:
        raise RuntimeError(
            "Expected RecA target not found.\n"
            "Please inspect recA_target_search_hits.csv"
        )

    return match.iloc[0]


# ============================================================
# FETCH BIOACTIVITY DATA
# ============================================================

def fetch_activities(target_chembl_id: str) -> pd.DataFrame:
    """Fetch raw activity data."""

    fields = [
        "activity_id",
        "assay_chembl_id",
        "assay_description",
        "canonical_smiles",
        "document_chembl_id",
        "molecule_chembl_id",
        "pchembl_value",
        "published_type",
        "published_units",
        "published_value",
        "standard_relation",
        "standard_type",
        "standard_units",
        "standard_value",
        "target_chembl_id",
    ]

    records = list(
        new_client.activity.filter(
            target_chembl_id=target_chembl_id
        ).only(fields)
    )

    if not records:
        raise RuntimeError(
            f"No activities found for target: {target_chembl_id}"
        )

    return pd.DataFrame(records)


# ============================================================
# CHUNK GENERATOR
# ============================================================

def chunked(values: list[str], size: int) -> Iterable[list[str]]:
    for idx in range(0, len(values), size):
        yield values[idx: idx + size]


# ============================================================
# FETCH MOLECULE METADATA
# ============================================================

def fetch_molecule_metadata(
    molecule_ids: list[str]
) -> pd.DataFrame:

    if not molecule_ids:
        return pd.DataFrame()

    fields = [
        "molecule_chembl_id",
        "pref_name",
        "max_phase",
        "molecule_type",
        "structure_type",
        "molecule_structures",
        "molecule_properties",
    ]

    all_records = []

    for batch in chunked(molecule_ids, MOLECULE_BATCH_SIZE):

        batch_records = list(
            new_client.molecule.filter(
                molecule_chembl_id__in=batch
            ).only(fields)
        )

        all_records.extend(batch_records)

    rows = []

    for record in all_records:

        structures = record.get("molecule_structures") or {}
        properties = record.get("molecule_properties") or {}

        rows.append({
            "molecule_chembl_id": record.get("molecule_chembl_id"),
            "molecule_pref_name": record.get("pref_name"),
            "max_phase": record.get("max_phase"),
            "molecule_type": record.get("molecule_type"),
            "structure_type": record.get("structure_type"),
            "canonical_smiles": structures.get("canonical_smiles"),
            "standard_inchi_key": structures.get("standard_inchi_key"),
            "full_mwt": properties.get("full_mwt"),
            "alogp": properties.get("alogp"),
            "psa": properties.get("psa"),
            "hba": properties.get("hba"),
            "hbd": properties.get("hbd"),
            "rtb": properties.get("rtb"),
            "aromatic_rings": properties.get("aromatic_rings"),
            "cx_logp": properties.get("cx_logp"),
        })

    return pd.DataFrame(rows).drop_duplicates(
        subset=["molecule_chembl_id"]
    )


# ============================================================
# PREPARE EC50 TABLES
# ============================================================

def prepare_activity_tables(
    raw_df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:

    df = raw_df.copy()

    df["standard_value_nM"] = pd.to_numeric(
        df["standard_value"],
        errors="coerce"
    )

    df["pchembl_value"] = pd.to_numeric(
        df["pchembl_value"],
        errors="coerce"
    )

    filtered = df[
        (df["standard_type"] == "EC50")
        &
        (df["standard_units"] == "nM")
        &
        (df["standard_relation"] == "=")
        &
        df["standard_value_nM"].notna()
        &
        (df["standard_value_nM"] > 0)
    ].copy()

    filtered["pEC50"] = filtered[
        "standard_value_nM"
    ].map(lambda x: 9 - math.log10(x))

    filtered["activity_label"] = filtered[
        "standard_value_nM"
    ].map(
        lambda x:
        "potent" if x <= 1000
        else "weak_or_moderate"
    )

    summary = (
        filtered.groupby(
            "molecule_chembl_id",
            as_index=False
        )
        .agg(
            n_records=("activity_id", "count"),
            median_ec50_nM=("standard_value_nM", "median"),
            min_ec50_nM=("standard_value_nM", "min"),
            max_ec50_nM=("standard_value_nM", "max"),
            median_pEC50=("pEC50", "median"),
        )
        .sort_values(
            ["median_pEC50", "min_ec50_nM"],
            ascending=[False, True]
        )
    )

    summary["activity_label"] = summary[
        "median_ec50_nM"
    ].map(
        lambda x:
        "potent" if x <= 1000
        else "weak_or_moderate"
    )

    return filtered, summary


# ============================================================
# SAVE OUTPUTS
# ============================================================

def save_outputs(
    output_dir: Path,
    target_hits: pd.DataFrame,
    selected_target: pd.Series,
    raw_activities: pd.DataFrame,
    filtered_activities: pd.DataFrame,
    activity_summary: pd.DataFrame,
    molecule_metadata: pd.DataFrame,
) -> None:

    output_dir.mkdir(parents=True, exist_ok=True)

    target_hits.to_csv(
        output_dir / "recA_target_search_hits.csv",
        index=False
    )

    pd.DataFrame(
        [selected_target.to_dict()]
    ).to_csv(
        output_dir / "recA_selected_target.csv",
        index=False
    )

    raw_activities.to_csv(
        output_dir / "recA_activities_raw.csv",
        index=False
    )

    filtered_activities.to_csv(
        output_dir / "recA_activities_ec50_exact.csv",
        index=False
    )

    activity_summary.to_csv(
        output_dir / "recA_activity_summary_ml.csv",
        index=False
    )

    molecule_metadata.to_csv(
        output_dir / "recA_molecule_metadata.csv",
        index=False
    )

    ml_dataset = activity_summary.merge(
        molecule_metadata,
        on="molecule_chembl_id",
        how="left"
    )

    ml_dataset.to_csv(
        output_dir / "recA_ml_dataset.csv",
        index=False
    )


# ============================================================
# MAIN
# ============================================================

def main(
    query: str = TARGET_SEARCH_QUERY,
    output_dir: str | Path = "outputs/recA_chembl"
) -> None:
    """Run the full ChEMBL download workflow.

    In this notebook version, arguments are passed directly to main()
    instead of using command-line argparse.
    """

    output_dir = Path(output_dir)

    print("Searching target...")
    target_hits = search_targets(query)

    print("Selecting RecA target...")
    selected_target = select_target(target_hits)

    print("Downloading activity data...")
    raw_activities = fetch_activities(
        selected_target["target_chembl_id"]
    )

    print("Preparing EC50 tables...")
    filtered_activities, activity_summary = (
        prepare_activity_tables(raw_activities)
    )

    molecule_ids = (
        activity_summary["molecule_chembl_id"]
        .dropna()
        .unique()
        .tolist()
    )

    print("Downloading molecule metadata...")
    molecule_metadata = fetch_molecule_metadata(
        molecule_ids
    )

    print("Saving outputs...")
    save_outputs(
        output_dir=output_dir,
        target_hits=target_hits,
        selected_target=selected_target,
        raw_activities=raw_activities,
        filtered_activities=filtered_activities,
        activity_summary=activity_summary,
        molecule_metadata=molecule_metadata,
    )

    print("\nWorkflow completed successfully.")
    print(f"Output folder: {output_dir}")


### Jalankan bagian download ChEMBL

Cell ini membutuhkan koneksi internet karena mengambil data langsung dari ChEMBL.

In [4]:
main()
# atau gunakan output folder khusus:
# main(output_dir="outputs/recA_chembl")

Searching target...
Selecting RecA target...
Preparing EC50 tables...
Saving outputs...

Workflow completed successfully.
Output folder: outputs/recA_chembl


## 2. Data Processing dan Binary Labels

Bagian ini membaca file `outputs/recA_chembl/recA_activities_ec50_exact.csv`, menghapus missing values, menghapus duplikat SMILES, memberi label active/intermediate/inactive, lalu membuat dataset binary active vs inactive.

In [5]:
from __future__ import annotations

from pathlib import Path
import pandas as pd


SCRIPT_DIR = Path.cwd()  # notebook working directory

INPUT_FILE = SCRIPT_DIR / "outputs" / "recA_chembl" / "recA_activities_ec50_exact.csv"
OUTPUT_DIR = SCRIPT_DIR / "outputs" / "recA_chembl"

ACTIVE_THRESHOLD_NM = 1000
INACTIVE_THRESHOLD_NM = 10000


def load_input() -> pd.DataFrame:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"\nInput file not found:\n{INPUT_FILE}\n\n"
            "Please run 01a_ChEMBL_download_workflow.py first."
        )

    return pd.read_csv(INPUT_FILE)


def process_dataframe(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    required_columns = ["molecule_chembl_id", "canonical_smiles", "standard_value"]
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    df = df.copy()
    df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")

    df2 = df[
        df["standard_value"].notna()
        & df["canonical_smiles"].notna()
    ].copy()

    df2["normalized_smiles"] = df2["canonical_smiles"]

    df2_nr = df2.drop_duplicates(subset=["normalized_smiles"]).copy()

    selection = ["molecule_chembl_id", "canonical_smiles", "standard_value"]
    df3 = df2_nr[selection].copy()

    return df2, df2_nr, df3


def label_bioactivity(df3: pd.DataFrame) -> pd.DataFrame:
    df4 = df3.copy()

    df4["bioactivity_class"] = "intermediate"

    df4.loc[
        df4["standard_value"] <= ACTIVE_THRESHOLD_NM,
        "bioactivity_class"
    ] = "active"

    df4.loc[
        df4["standard_value"] >= INACTIVE_THRESHOLD_NM,
        "bioactivity_class"
    ] = "inactive"

    return df4


def create_binary_dataset(df4: pd.DataFrame) -> pd.DataFrame:
    df5 = df4[df4["bioactivity_class"] != "intermediate"].copy()

    df5["bioactivity_numeric"] = df5["bioactivity_class"].map(
        {"inactive": 0, "active": 1}
    ).astype(int)

    return df5


def save_outputs(
    df2: pd.DataFrame,
    df2_nr: pd.DataFrame,
    df3: pd.DataFrame,
    df4: pd.DataFrame,
    df5: pd.DataFrame,
) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df2.to_csv(OUTPUT_DIR / "recA_ec50_nonnull.csv", index=False)
    df2_nr.to_csv(OUTPUT_DIR / "recA_ec50_deduplicated.csv", index=False)
    df3.to_csv(OUTPUT_DIR / "recA_ec50_final_selection.csv", index=False)
    df4.to_csv(OUTPUT_DIR / "recA_ec50_labeled.csv", index=False)
    df5.to_csv(OUTPUT_DIR / "recA_ec50_binary.csv", index=False)


def main() -> None:
    df = load_input()

    df2, df2_nr, df3 = process_dataframe(df)
    df4 = label_bioactivity(df3)
    df5 = create_binary_dataset(df4)

    save_outputs(df2, df2_nr, df3, df4, df5)

    print("Target: CHEMBL1741171 | Protein RecA | Mycobacterium tuberculosis")
    print(f"Input rows: {len(df)}")
    print(f"After removing missing standard_value and canonical_smiles: {len(df2)}")
    print(f"After removing duplicate normalized_smiles: {len(df2_nr)}")
    print(f"Unique normalized SMILES: {df2_nr['normalized_smiles'].nunique()}")

    print(
        f"Label thresholds: active <= {ACTIVE_THRESHOLD_NM} nM, "
        f"intermediate = {ACTIVE_THRESHOLD_NM} < x < {INACTIVE_THRESHOLD_NM} nM, "
        f"inactive >= {INACTIVE_THRESHOLD_NM} nM"
    )

    print("\nBioactivity class counts:")
    print(df4["bioactivity_class"].value_counts().to_string())

    print("\nBinary dataset counts:")
    print(df5["bioactivity_class"].value_counts().to_string())

    print("\nOutput saved to:")
    print(OUTPUT_DIR)


### Jalankan processing binary labels

In [6]:
main()

Target: CHEMBL1741171 | Protein RecA | Mycobacterium tuberculosis
Input rows: 1914
After removing missing standard_value and canonical_smiles: 1912
After removing duplicate normalized_smiles: 1907
Unique normalized SMILES: 1907
Label thresholds: active <= 1000 nM, intermediate = 1000 < x < 10000 nM, inactive >= 10000 nM

Bioactivity class counts:
bioactivity_class
intermediate    992
inactive        601
active          314

Binary dataset counts:
bioactivity_class
inactive    601
active      314

Output saved to:
/content/outputs/recA_chembl


## 3. Q1-style Data Curation dan Balanced Dataset

Bagian ini melakukan curation final dengan aturan active ≤ 1000 nM, inactive ≥ 10000 nM, menghapus intermediate untuk dataset binary, dan membuat balanced dataset 50:50.

In [7]:
"""Q1-style RecA-TB data curation workflow.

Biological target:
- Mycobacterium tuberculosis RecA (ChEMBL target CHEMBL1741171), not the
  human homolog RAD51.

Methodological basis:
- ChEMBL as curated bioactivity source:
  Zdrazil et al., Nucleic Acids Research (2024), doi:10.1093/nar/gkad1004
- MtRecA as bacterial DNA-repair target:
  Datta et al., Nucleic Acids Research (2000), doi:10.1093/nar/28.24.4964
- RecA as antibacterial drug target:
  Pavlopoulou, Frontiers in Bioscience (2018), doi:10.2741/4580

Data handling rules used here:
- exact EC50 records only, in line with the available local RecA assay table;
- remove rows with missing standard_value or canonical_smiles;
- deduplicate by canonical_smiles to avoid repeated chemistry in modeling;
- keep active/intermediate/inactive labels explicit before building the binary set.

The active/intermediate/inactive cutoffs follow a common QSAR screening
convention used in international bioactivity-classification studies:
- active <= 1000 nM
- inactive >= 10000 nM
- intermediate otherwise
"""

from __future__ import annotations

from pathlib import Path
import pandas as pd


SCRIPT_DIR = Path.cwd()  # notebook working directory

INPUT_FILE = SCRIPT_DIR / "recA_activities_ec50_exact.csv"
OUTPUT_DIR = SCRIPT_DIR / "outputs"

ACTIVE_THRESHOLD_NM = 1000
INACTIVE_THRESHOLD_NM = 10000
RANDOM_STATE = 42


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"\nInput file not found:\n{INPUT_FILE}\n\n"
            "Make sure 'recA_activities_ec50_exact.csv' is in the same folder as this script."
        )

    df = pd.read_csv(INPUT_FILE)

    required_columns = ["molecule_chembl_id", "canonical_smiles", "standard_value"]
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")

    df_nonnull = df[
        df["standard_value"].notna()
        & df["canonical_smiles"].notna()
    ].copy()

    df_nonnull["normalized_smiles"] = df_nonnull["canonical_smiles"]

    df_dedup = df_nonnull.drop_duplicates(subset=["normalized_smiles"]).copy()

    curated = df_dedup[
        ["molecule_chembl_id", "canonical_smiles", "standard_value"]
    ].copy()

    curated["bioactivity_class"] = "intermediate"

    curated.loc[
        curated["standard_value"] <= ACTIVE_THRESHOLD_NM,
        "bioactivity_class"
    ] = "active"

    curated.loc[
        curated["standard_value"] >= INACTIVE_THRESHOLD_NM,
        "bioactivity_class"
    ] = "inactive"

    binary = curated[
        curated["bioactivity_class"].isin(["active", "inactive"])
    ].copy()

    binary["class"] = binary["bioactivity_class"].map(
        {"inactive": 0, "active": 1}
    ).astype(int)

    active = binary[binary["bioactivity_class"] == "active"].copy()
    inactive = binary[binary["bioactivity_class"] == "inactive"].copy()

    target_n = min(len(active), len(inactive))

    if target_n == 0:
        raise ValueError(
            "Cannot create balanced dataset because active or inactive class is empty."
        )

    active_balanced = active.sample(n=target_n, random_state=RANDOM_STATE)
    inactive_balanced = inactive.sample(n=target_n, random_state=RANDOM_STATE)

    balanced = (
        pd.concat([active_balanced, inactive_balanced], axis=0)
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )

    df_nonnull.to_csv(OUTPUT_DIR / "01_recA_ec50_nonnull.csv", index=False)
    df_dedup.to_csv(OUTPUT_DIR / "01_recA_ec50_deduplicated.csv", index=False)
    curated.to_csv(OUTPUT_DIR / "01_recA_ec50_labeled.csv", index=False)
    binary.to_csv(OUTPUT_DIR / "01_recA_ec50_binary.csv", index=False)
    balanced.to_csv(
        OUTPUT_DIR / "01_recA_ec50_binary_balanced_50_50.csv",
        index=False
    )

    summary = pd.DataFrame(
        [
            {"step": "Raw exact EC50 records", "records": len(df)},
            {"step": "After removing missing values", "records": len(df_nonnull)},
            {"step": "After SMILES deduplication", "records": len(df_dedup)},
            {
                "step": "Active compounds",
                "records": int((curated["bioactivity_class"] == "active").sum()),
            },
            {
                "step": "Intermediate compounds",
                "records": int((curated["bioactivity_class"] == "intermediate").sum()),
            },
            {
                "step": "Inactive compounds",
                "records": int((curated["bioactivity_class"] == "inactive").sum()),
            },
            {"step": "Final binary dataset", "records": len(binary)},
            {
                "step": "Balanced active instances",
                "records": int((balanced["bioactivity_class"] == "active").sum()),
            },
            {
                "step": "Balanced inactive instances",
                "records": int((balanced["bioactivity_class"] == "inactive").sum()),
            },
            {"step": "Balanced modeling dataset", "records": len(balanced)},
        ]
    )

    summary.to_csv(OUTPUT_DIR / "01_data_curation_summary.csv", index=False)

    print(summary.to_string(index=False))
    print(f"\nInput file used:\n{INPUT_FILE}")
    print(f"\nOutput files saved to:\n{OUTPUT_DIR}")


### Jalankan curation final

Catatan: script ketiga secara default mencari file `recA_activities_ec50_exact.csv` di folder kerja notebook. Jika Anda ingin memakai output dari bagian ChEMBL, copy file tersebut dari `outputs/recA_chembl/` ke folder kerja, atau ubah `INPUT_FILE` di cell sebelumnya.

In [9]:
from pathlib import Path

# Jika file input ada di outputs/recA_chembl, jalankan baris ini sebelum main():
INPUT_FILE = Path.cwd() / 'outputs' / 'recA_chembl' / 'recA_activities_ec50_exact.csv'
OUTPUT_DIR = Path.cwd() / 'outputs' / 'recA_chembl'

main()

                         step  records
       Raw exact EC50 records     1914
After removing missing values     1912
   After SMILES deduplication     1907
             Active compounds      314
       Intermediate compounds      992
           Inactive compounds      601
         Final binary dataset      915
    Balanced active instances      314
  Balanced inactive instances      314
    Balanced modeling dataset      628

Input file used:
/content/outputs/recA_chembl/recA_activities_ec50_exact.csv

Output files saved to:
/content/outputs/recA_chembl


## Output penting

Setelah semua cell dijalankan, file output utama biasanya berada di folder `outputs/recA_chembl`, termasuk:

- `recA_activities_raw.csv`
- `recA_activities_ec50_exact.csv`
- `recA_activity_summary_ml.csv`
- `recA_molecule_metadata.csv`
- `recA_ml_dataset.csv`
- `recA_ec50_labeled.csv`
- `recA_ec50_binary.csv`
- dataset balanced jika bagian final dijalankan.